In [1]:
# =============================================================================
# CdSe Quantum Dot Calculator – Jupyter Launcher (No Unicode Errors)
# Website Title: Piyasee Khokhar
# =============================================================================
import subprocess
import sys
import time
import webbrowser
import os

# Step 1: Install streamlit if not present
try:
    import streamlit
except ImportError:
    print("Installing streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])
    print("Streamlit installed.")

# Step 2: Write the complete app.py file (ASCII only)
app_code = '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CdSe Quantum Dot Calculator – Interactive Dashboard (Complete)
Website: Piyasee Khokhar
"""

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================================
# 1. CONSTANTS & CORE FUNCTIONS
# ============================================================================
E_g_bulk = 1.74
me_star = 0.13
mh_star = 0.45
eps_r = 9.5
a_B = 5.6
CBM_bulk = -4.95
VBM_bulk = -6.69
A_conf = 3.729
B_coul = 0.2707

def band_gap(R):
    return E_g_bulk + A_conf / R**2 - B_coul / R

def cbm(R):
    delta_Eg = band_gap(R) - E_g_bulk
    delta_Ec = (mh_star / (me_star + mh_star)) * delta_Eg
    return CBM_bulk + delta_Ec

def vbm(R):
    delta_Eg = band_gap(R) - E_g_bulk
    delta_Ev = (me_star / (me_star + mh_star)) * delta_Eg
    return VBM_bulk - delta_Ev

def absorption_edge(R):
    return 1240 / band_gap(R)

def blue_shift(R):
    lambda_bulk = 1240 / E_g_bulk
    return lambda_bulk - absorption_edge(R)

def surface_fraction(R):
    d_atom = 0.43
    return min(1.0, 3 * d_atom / R)

def volume(R):
    return (4/3) * np.pi * R**3

def surface_area(R):
    return 4 * np.pi * R**2

def sv_ratio(R):
    return 3 / R

def exciton_ratio(R):
    return R / a_B

def regime(R):
    ratio = R / a_B
    if ratio < 1:
        return "Strong"
    elif ratio <= 1.1:
        return "Intermediate"
    else:
        return "Weak"

def number_of_formula_units(R):
    rho = 5.81
    M = 191.37
    NA = 6.02214076e23
    V_cm3 = volume(R) * 1e-21
    return int((V_cm3 * rho * NA) / M)

def total_atom_count(R):
    return 2 * number_of_formula_units(R)

def confinement_energy(R):
    return A_conf / R**2

def coulomb_energy(R):
    return B_coul / R

# ============================================================================
# 2. EXPERIMENTAL DATA
# ============================================================================
exp_data = [
    (2.0, 2.40, "Murray et al. 1993"),
    (2.3, 2.20, "Peng et al. 1997"),
    (2.8, 2.02, "Murray et al. 1993"),
    (3.0, 2.00, "Norris & Bawendi 1996"),
    (3.5, 1.95, "Norris & Bawendi 1996"),
    (4.0, 1.90, "Murray et al. 1993"),
    (5.0, 1.83, "Peng et al. 1997"),
    (6.0, 1.79, "Norris & Bawendi 1996"),
    (8.0, 1.76, "Murray et al. 1993"),
    (10.0, 1.74, "NSM Archive (bulk)")
]
exp_R = np.array([d[0] for d in exp_data])
exp_Eg = np.array([d[1] for d in exp_data])

# ============================================================================
# 3. PLOTTING FUNCTIONS (all 21 graphs)
# ============================================================================
def plot_sv(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, 3/R_plot, 'b-', linewidth=2)
    ax.scatter(R, 3/R, color='red', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("S/V (nm-1)")
    ax.set_title("1. Surface-to-Volume Ratio")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_atoms(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    atoms = [number_of_formula_units(r) for r in R_plot]
    ax.plot(R_plot, atoms, 'g-', linewidth=2)
    ax.scatter(R, number_of_formula_units(R), color='red', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("CdSe Units")
    ax.set_title("2. Number of CdSe Units")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_surf_frac(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    sf = [surface_fraction(r) for r in R_plot]
    ax.plot(R_plot, sf, 'r-', linewidth=2)
    ax.scatter(R, surface_fraction(R), color='blue', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Surface Fraction")
    ax.set_title("3. Surface Atom Fraction")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_conf(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, confinement_energy(R_plot), 'purple', linewidth=2)
    ax.scatter(R, confinement_energy(R), color='orange', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("E_conf (eV)")
    ax.set_title("4. Confinement Energy")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_coul(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, coulomb_energy(R_plot), 'orange', linewidth=2)
    ax.scatter(R, coulomb_energy(R), color='purple', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("E_coul (eV)")
    ax.set_title("5. Coulomb Energy")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_bandgap_exp(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    Eg_plot = [band_gap(r) for r in R_plot]
    ax.plot(R_plot, Eg_plot, 'b-', linewidth=2, label='Brus model')
    ax.scatter(exp_R, exp_Eg, color='red', marker='x', s=80, label='Experimental')
    ax.scatter(R, band_gap(R), color='green', marker='o', s=120, label=f'R={R:.1f} nm', zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Band Gap (eV)")
    ax.set_title("6. Size-Dependent Band Gap")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_shift(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [band_gap(r)-E_g_bulk for r in R_plot], 'brown', linewidth=2)
    ax.scatter(R, band_gap(R)-E_g_bulk, color='green', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Delta Eg (eV)")
    ax.set_title("7. Band Gap Widening")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_cbm_vbm(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [cbm(r) for r in R_plot], 'b-', label='CBM')
    ax.plot(R_plot, [vbm(r) for r in R_plot], 'r-', label='VBM')
    ax.scatter(R, cbm(R), color='blue', s=80); ax.scatter(R, vbm(R), color='red', s=80)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Energy (eV) – Vacuum=0")
    ax.set_title("8. CBM and VBM")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_rab(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [r/a_B for r in R_plot], 'darkcyan', linewidth=2)
    ax.axhline(y=1, color='k', linestyle='--', label='R/aB=1')
    ax.scatter(R, exciton_ratio(R), color='magenta', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("R/aB")
    ax.set_title("9. Exciton Confinement Ratio")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_abs_edge(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [absorption_edge(r) for r in R_plot], 'm-', linewidth=2)
    ax.scatter(R, absorption_edge(R), color='cyan', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Absorption Edge (nm)")
    ax.set_title("10. Absorption Edge")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_blue_shift(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [blue_shift(r) for r in R_plot], 'olive', linewidth=2)
    ax.scatter(R, blue_shift(R), color='brown', s=80, zorder=5)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Blue Shift (nm)")
    ax.set_title("11. Blue Shift")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_delta_shifts(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    dEc = [(mh_star/(me_star+mh_star))*(band_gap(r)-E_g_bulk) for r in R_plot]
    dEv = [(me_star/(me_star+mh_star))*(band_gap(r)-E_g_bulk) for r in R_plot]
    ax.plot(R_plot, dEc, 'b-', label='Delta Ec')
    ax.plot(R_plot, dEv, 'orange', label='Delta Ev')
    ax.scatter(R, (mh_star/(me_star+mh_star))*(band_gap(R)-E_g_bulk), color='blue', s=80)
    ax.scatter(R, (me_star/(me_star+mh_star))*(band_gap(R)-E_g_bulk), color='orange', s=80)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Energy Shift (eV)")
    ax.set_title("12. Band Edge Shifts")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_conf_vs_coul(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    ax.plot(R_plot, [confinement_energy(r) for r in R_plot], 'b-', label='Confinement')
    ax.plot(R_plot, [coulomb_energy(r) for r in R_plot], 'r-', label='Coulomb')
    ax.scatter(R, confinement_energy(R), color='blue', s=80)
    ax.scatter(R, coulomb_energy(R), color='red', s=80)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Energy (eV)")
    ax.set_title("13. Confinement vs Coulomb")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_volume_atoms(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    vol = [volume(r) for r in R_plot]
    atoms = [number_of_formula_units(r) for r in R_plot]
    ax.plot(vol, atoms, 'g-', linewidth=2)
    ax.scatter(volume(R), number_of_formula_units(R), color='red', s=80, zorder=5)
    ax.set_xlabel("Volume (nm3)"); ax.set_ylabel("CdSe Units")
    ax.set_title("14. Volume vs CdSe Units")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_energy_evo(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    cbm_vals = [cbm(r) for r in R_plot]
    vbm_vals = [vbm(r) for r in R_plot]
    ax.fill_between(R_plot, vbm_vals, cbm_vals, alpha=0.3, color='gray')
    ax.plot(R_plot, cbm_vals, 'b-', label='CBM')
    ax.plot(R_plot, vbm_vals, 'r-', label='VBM')
    ax.scatter(R, cbm(R), color='blue', s=80)
    ax.scatter(R, vbm(R), color='red', s=80)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Energy (eV) – Vacuum=0")
    ax.set_title("15. Energy Level Evolution")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_eg_vs_1r2(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    invR2 = 1 / R_plot**2
    Eg_plot = [band_gap(r) for r in R_plot]
    ax.plot(invR2, Eg_plot, 'navy', linewidth=2)
    ax.scatter(1/R**2, band_gap(R), color='red', s=80, zorder=5)
    coeffs = np.polyfit(invR2, Eg_plot, 1)
    fit_line = np.polyval(coeffs, invR2)
    ax.plot(invR2, fit_line, 'r--', label=f'Fit: Eg = {coeffs[0]:.2f}*(1/R2) + {coeffs[1]:.2f}')
    ax.set_xlabel("1/R2 (nm-2)"); ax.set_ylabel("Band Gap (eV)")
    ax.set_title("16. Band Gap vs 1/R2")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_bar_chart(R):
    fig, ax = plt.subplots(figsize=(6,4))
    sel_R = [1, 3, 5, 10]
    cbm_sel = [cbm(r) for r in sel_R]
    vbm_sel = [vbm(r) for r in sel_R]
    x_pos = np.arange(len(sel_R))
    width = 0.35
    ax.bar(x_pos - width/2, cbm_sel, width, label='CBM', color='blue', alpha=0.7)
    ax.bar(x_pos + width/2, vbm_sel, width, label='VBM', color='red', alpha=0.7)
    ax.set_xticks(x_pos); ax.set_xticklabels([f"R={r} nm" for r in sel_R])
    ax.set_ylabel("Energy (eV) – Vacuum=0")
    ax.set_title("17. CBM/VBM at Selected Radii")
    ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.6)
    return fig

def plot_validation_scatter(R):
    fig, ax = plt.subplots(figsize=(6,4))
    theory = [band_gap(r) for r in exp_R]
    ax.scatter(theory, exp_Eg, color='red', s=80, label='Literature')
    min_val = min(min(theory), min(exp_Eg))
    max_val = max(max(theory), max(exp_Eg))
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', label='y=x')
    for i, r in enumerate(exp_R):
        ax.annotate(f"{r:.1f} nm", (theory[i], exp_Eg[i]), fontsize=8, xytext=(5,5), textcoords='offset points')
    ax.set_xlabel("Theoretical Eg (eV)"); ax.set_ylabel("Experimental Eg (eV)")
    ax.set_title("18. Theory vs Experiment")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_percent_error(R):
    fig, ax = plt.subplots(figsize=(6,4))
    theory = [band_gap(r) for r in exp_R]
    rel_err = 100 * np.abs(np.array(theory) - exp_Eg) / exp_Eg
    ax.bar(exp_R, rel_err, width=0.3, color='coral', edgecolor='black')
    ax.axhline(y=5, color='r', linestyle='--', label='5% threshold')
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Relative Error (%)")
    ax.set_title("19. Percentage Error")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_error_bars(R):
    fig, ax = plt.subplots(figsize=(6,4))
    R_plot = np.linspace(1, 10, 100)
    Eg_plot = [band_gap(r) for r in R_plot]
    ax.plot(R_plot, Eg_plot, 'b-', linewidth=2, label='Brus')
    ax.errorbar(exp_R, exp_Eg, yerr=0.03, fmt='rx', capsize=4, label='Experimental')
    ax.scatter(R, band_gap(R), color='green', s=80, label=f'Selected R={R:.1f} nm')
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Band Gap (eV)")
    ax.set_title("20. Comparison with Error Bars")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    return fig

def plot_residuals(R):
    fig, ax = plt.subplots(figsize=(6,4))
    res = [band_gap(r) - exp_Eg[i] for i, r in enumerate(exp_R)]
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.8)
    ax.bar(exp_R, res, width=0.3, color='green', alpha=0.7)
    ax.set_xlabel("Radius (nm)"); ax.set_ylabel("Residual (eV)")
    ax.set_title("21. Residual Plot")
    ax.grid(True, linestyle='--', alpha=0.6)
    return fig

# ============================================================================
# 4. STREAMLIT LAYOUT
# ============================================================================
st.set_page_config(page_title="Piyasee Khokhar", layout="wide")
st.title("CdSe Quantum Dot Calculator")
st.markdown("**Piyasee Khokhar | GC University Hyderabad**")
st.markdown("**Interactive dashboard for size-dependent properties**")
st.markdown("---")

# Sidebar
with st.sidebar:
    st.header("Input")
    R = st.slider("Quantum Dot Radius (nm)", 1.0, 10.0, 3.0, 0.1)
    st.markdown("---")
    st.header("Material Constants")
    st.write(f"**Bulk Eg:** {E_g_bulk} eV")
    st.write(f"**me*:** {me_star} m0")
    st.write(f"**mh*:** {mh_star} m0")
    st.write(f"**eps_r:** {eps_r}")
    st.write(f"**a_B:** {a_B} nm")
    st.write(f"**CBM(bulk):** {CBM_bulk} eV")
    st.write(f"**VBM(bulk):** {VBM_bulk} eV")

# Top metrics
col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Band Gap (Eg)", f"{band_gap(R):.4f} eV")
with col2:
    st.metric("CBM", f"{cbm(R):.2f} eV")
with col3:
    st.metric("VBM", f"{vbm(R):.2f} eV")
with col4:
    st.metric("Absorption Edge", f"{absorption_edge(R):.1f} nm")

st.markdown("---")
with st.expander("Full Parameter List", expanded=True):
    cols = st.columns(3)
    params = {
        "Radius": f"{R:.1f} nm",
        "Confinement Energy": f"{confinement_energy(R):.4f} eV",
        "Coulomb Energy": f"{coulomb_energy(R):.4f} eV",
        "Blue Shift": f"{blue_shift(R):.1f} nm",
        "Surface Fraction": f"{surface_fraction(R):.2f}",
        "Volume": f"{volume(R):.1f} nm3",
        "Surface Area": f"{surface_area(R):.1f} nm2",
        "S/V Ratio": f"{sv_ratio(R):.3f} nm-1",
        "CdSe Units": f"{number_of_formula_units(R)}",
        "Total Atoms": f"{total_atom_count(R)}",
        "R/aB": f"{exciton_ratio(R):.2f}",
        "Regime": regime(R)
    }
    for i, (key, val) in enumerate(params.items()):
        cols[i % 3].metric(key, val)

st.markdown("---")
st.header("Graphical Analysis (21 Graphs)")

# List of all plotting functions with titles
plot_funcs = [
    ("Surface-to-Volume Ratio", plot_sv),
    ("Number of CdSe Units", plot_atoms),
    ("Surface Atom Fraction", plot_surf_frac),
    ("Confinement Energy", plot_conf),
    ("Coulomb Energy", plot_coul),
    ("Size-Dependent Band Gap", plot_bandgap_exp),
    ("Band Gap Widening", plot_shift),
    ("CBM and VBM", plot_cbm_vbm),
    ("Exciton Confinement Ratio", plot_rab),
    ("Absorption Edge", plot_abs_edge),
    ("Blue Shift", plot_blue_shift),
    ("Band Edge Shifts", plot_delta_shifts),
    ("Confinement vs Coulomb", plot_conf_vs_coul),
    ("Volume vs CdSe Units", plot_volume_atoms),
    ("Energy Level Evolution", plot_energy_evo),
    ("Band Gap vs 1/R2", plot_eg_vs_1r2),
    ("CBM/VBM at Selected Radii", plot_bar_chart),
    ("Theory vs Experiment", plot_validation_scatter),
    ("Percentage Error", plot_percent_error),
    ("Comparison with Error Bars", plot_error_bars),
    ("Residual Plot", plot_residuals)
]

# Display graphs in a grid (3 columns)
for i in range(0, len(plot_funcs), 3):
    cols = st.columns(3)
    for j, (title, func) in enumerate(plot_funcs[i:i+3]):
        with cols[j]:
            st.pyplot(func(R))
            plt.close()

st.markdown("---")

# Dataset table
with st.expander("Complete Dataset (R = 1-10 nm)", expanded=False):
    radii = np.arange(1.0, 10.5, 1.0)
    data = []
    for r in radii:
        data.append({
            "R (nm)": r,
            "Eg (eV)": band_gap(r),
            "CBM (eV)": cbm(r),
            "VBM (eV)": vbm(r),
            "lambda_edge (nm)": absorption_edge(r),
            "Blue Shift (nm)": blue_shift(r),
            "S/V (nm-1)": sv_ratio(r),
            "Surf. Fraction": surface_fraction(r),
            "Regime": regime(r)
        })
    df = pd.DataFrame(data)
    st.dataframe(df.style.format({
        "Eg (eV)": "{:.4f}",
        "CBM (eV)": "{:.2f}",
        "VBM (eV)": "{:.2f}",
        "lambda_edge (nm)": "{:.1f}",
        "Blue Shift (nm)": "{:.1f}",
        "S/V (nm-1)": "{:.3f}",
        "Surf. Fraction": "{:.2f}"
    }))
    csv = df.to_csv(index=False)
    st.download_button("Download CSV", csv, "CdSe_QD_dataset.csv", "text/csv")

st.caption("Piyasee Khokhar | GC University Hyderabad | All parameters verified against literature")
'''

# Write app.py with UTF-8 encoding
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py has been created with Piyasee Khokhar as the website author.")

# Step 3: Launch the Streamlit app
print("Launching Streamlit app...")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                         "--server.headless", "true"])
time.sleep(5)
webbrowser.open("http://localhost:8501")
print("🌐 Open in your browser: http://localhost:8501")
print("⏳ Press Ctrl+C in the terminal to stop the server when done.")

✅ app.py has been created with Piyasee Khokhar as the website author.
Launching Streamlit app...
🌐 Open in your browser: http://localhost:8501
⏳ Press Ctrl+C in the terminal to stop the server when done.
